In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [7]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [8]:
X = df.drop("Class" , axis =1)
y = df["Class"]


In [9]:
from sklearn.preprocessing import StandardScaler , LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)



In [10]:
from sklearn.model_selection import train_test_split

X_train , X_test , y_train , y_test  = train_test_split(
    X , y , test_size = 0.2 , random_state = 42)



In [11]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)




### ANN


In [13]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , TensorDataset


In [15]:

X_train_tensor = torch.tensor(X_train_scaled , dtype = torch.float32)
y_train_tensor = torch.tensor(y_train , dtype= torch.long)

X_test_tensor = torch.tensor(X_test_scaled , dtype = torch.float32)
y_test_tensor = torch.tensor(y_test , dtype = torch.long)

In [16]:
train_dataset = TensorDataset(X_train_tensor , y_train_tensor)
test_dataset = TensorDataset(X_test_tensor , y_test_tensor)


In [17]:
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset , batch_size = 32)

### Build Your ANN

In [30]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 7)
        )

    # Keep forward() at the same indentation level as __init__()
    def forward(self, x):
        return self.model(x)

In [31]:
model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [32]:
# Training the NN
epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"Epoch = {epoch + 1}/{epochs}, Loss = {train_loss:.4f}")

Epoch = 1/100, Loss = 1.6665
Epoch = 2/100, Loss = 1.0750
Epoch = 3/100, Loss = 0.7133
Epoch = 4/100, Loss = 0.5421
Epoch = 5/100, Loss = 0.4556
Epoch = 6/100, Loss = 0.3929
Epoch = 7/100, Loss = 0.3417
Epoch = 8/100, Loss = 0.3067
Epoch = 9/100, Loss = 0.2739
Epoch = 10/100, Loss = 0.2539
Epoch = 11/100, Loss = 0.2343
Epoch = 12/100, Loss = 0.2249
Epoch = 13/100, Loss = 0.2093
Epoch = 14/100, Loss = 0.1982
Epoch = 15/100, Loss = 0.1965
Epoch = 16/100, Loss = 0.1807
Epoch = 17/100, Loss = 0.1722
Epoch = 18/100, Loss = 0.1654
Epoch = 19/100, Loss = 0.1664
Epoch = 20/100, Loss = 0.1564
Epoch = 21/100, Loss = 0.1530
Epoch = 22/100, Loss = 0.1512
Epoch = 23/100, Loss = 0.1420
Epoch = 24/100, Loss = 0.1336
Epoch = 25/100, Loss = 0.1406
Epoch = 26/100, Loss = 0.1373
Epoch = 27/100, Loss = 0.1309
Epoch = 28/100, Loss = 0.1281
Epoch = 29/100, Loss = 0.1248
Epoch = 30/100, Loss = 0.1216
Epoch = 31/100, Loss = 0.1127
Epoch = 32/100, Loss = 0.1119
Epoch = 33/100, Loss = 0.1066
Epoch = 34/100, Los

In [35]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)

        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("Total vals:", total)
print("Correct vals:", correct)
print("Accuracy:", 100 * correct / total, "%")


Total vals: 180
Correct vals: 169
Accuracy: 93.88888888888889 %
